In [ ]:
# In[1]:

import pandas as pd

# Load the telemetry data
metric_df = pd.read_csv('metric.csv')
log_df = pd.read_csv('log.csv')
trace_df = pd.read_csv('trace.csv')

# Convert timestamp to datetime
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Aggregate metrics by component and KPI
metric_agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95_value=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

log_agg = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95_value=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

trace_agg = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95_value=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Combine all aggregated data into a single DataFrame
combined_agg = pd.concat([metric_agg, log_agg, trace_agg], ignore_index=True)

# Display the combined aggregated results
combined_agg_summary = combined_agg.head(20)
combined_agg_summary

```
Out[1]:
```
The aggregation of KPIs from the telemetry data has been successfully completed. The results include various metrics for different components, such as CPU, memory, disk I/O, and latency. Each component-KPI combination has been analyzed, providing counts, minimum and maximum values, and the 95th percentile (P95) values. The earliest timestamps for each KPI have also been recorded. This summary allows for a quick assessment of the performance metrics across the system components.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count     min_value     max_value     p95_value        earliest_timestamp log_name trace_name
0          carts         cpu     25  1.170727e+00  7.958958e+01  7.934422e+01 2024-01-19 14:52:00+00:00      NaN        NaN
1          carts      diskio     25  0.000000e+00  1.163966e+05  5.924902e+04 2024-01-19 14:52:00+00:00      NaN        NaN
2          carts       error      5  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 15:12:00+00:00      NaN        NaN
3          carts  latency-50     25  1.266706e-02  2.043347e-02  1.948694e-02 2024-01-19 14:52:00+00:00      NaN        NaN
4          carts  latency-90     25  2.273448e-02  6.528074e-02  4.712160e-02 2024-01-19 14:52:00+00:00      NaN        NaN
5          carts         mem     25  2.162033e+08  2.224593e+08  2.219550e+08 2024-01-19 14:52:00+00:00      NaN        NaN
6          carts      socket     25  9.500000e+00  1.343333e+01  1.305333e+01 2024-01-19 14:52:00+00:00      NaN        NaN
7          carts    workload     25  7.641183e+00  8.655417e+00  8.486297e+00 2024-01-19 14:52:00+00:00      NaN        NaN
8       carts-db         cpu     25  3.034738e+00  4.388721e+00  4.275059e+00 2024-01-19 14:52:00+00:00      NaN        NaN
9       carts-db      diskio     25  1.790057e+06  2.296373e+06  2.270569e+06 2024-01-19 14:52:00+00:00      NaN        NaN
10      carts-db         mem     25  7.884390e+07  8.425316e+07  8.396829e+07 2024-01-19 14:52:00+00:00      NaN        NaN
11      carts-db      socket     25  6.000000e+00  7.000000e+00  7.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN
12     catalogue         cpu     25  1.336217e-01  1.858281e-01  1.833097e-01 2024-01-19 14:52:00+00:00      NaN        NaN
13     catalogue  latency-50     25  3.002940e-03  3.110457e-03  3.095734e-03 2024-01-19 14:52:00+00:00      NaN        NaN
14     catalogue  latency-90     25  4.605291e-03  4.798823e-03  4.772321e-03 2024-01-19 14:52:00+00:00      NaN        NaN
15     catalogue         mem     25  6.176768e+06  6.272137e+06  6.267372e+06 2024-01-19 14:52:00+00:00      NaN        NaN
16     catalogue      socket     25  6.000000e+00  6.000000e+00  6.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN
17     catalogue    workload     25  3.925483e+00  4.323467e+00  4.260883e+00 2024-01-19 14:52:00+00:00      NaN        NaN
18  catalogue-db         cpu     25  1.749846e-01  1.977208e-01  1.954904e-01 2024-01-19 14:52:00+00:00      NaN        NaN
19  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN```
```

In [ ]:
# In[2]:

# Define the incident window
incident_start = pd.to_datetime('2024-01-19 14:49:49', utc=True)
incident_end = pd.to_datetime('2024-01-19 15:19:49', utc=True)

# Filter the combined aggregated data for the incident window
filtered_agg = combined_agg[(combined_agg['earliest_timestamp'] >= incident_start) & 
                             (combined_agg['earliest_timestamp'] <= incident_end)]

# Display the filtered results
filtered_agg_summary = filtered_agg.head(20)
filtered_agg_summary

```
Out[2]:
```
The filtered results show the aggregated KPIs for various components during the incident window from 2024-01-19 14:49:49 to 2024-01-19 15:19:49 UTC. The data includes metrics such as CPU usage, disk I/O, memory usage, and latency for components like "carts," "carts-db," "catalogue," and "catalogue-db." Each entry provides the count of observations, minimum and maximum values, the 95th percentile (P95) values, and the earliest timestamp within the specified time frame. This information is crucial for identifying any anomalies that may have occurred during this period.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count     min_value     max_value     p95_value        earliest_timestamp log_name trace_name
0          carts         cpu     25  1.170727e+00  7.958958e+01  7.934422e+01 2024-01-19 14:52:00+00:00      NaN        NaN
1          carts      diskio     25  0.000000e+00  1.163966e+05  5.924902e+04 2024-01-19 14:52:00+00:00      NaN        NaN
2          carts       error      5  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 15:12:00+00:00      NaN        NaN
3          carts  latency-50     25  1.266706e-02  2.043347e-02  1.948694e-02 2024-01-19 14:52:00+00:00      NaN        NaN
4          carts  latency-90     25  2.273448e-02  6.528074e-02  4.712160e-02 2024-01-19 14:52:00+00:00      NaN        NaN
5          carts         mem     25  2.162033e+08  2.224593e+08  2.219550e+08 2024-01-19 14:52:00+00:00      NaN        NaN
6          carts      socket     25  9.500000e+00  1.343333e+01  1.305333e+01 2024-01-19 14:52:00+00:00      NaN        NaN
7          carts    workload     25  7.641183e+00  8.655417e+00  8.486297e+00 2024-01-19 14:52:00+00:00      NaN        NaN
8       carts-db         cpu     25  3.034738e+00  4.388721e+00  4.275059e+00 2024-01-19 14:52:00+00:00      NaN        NaN
9       carts-db      diskio     25  1.790057e+06  2.296373e+06  2.270569e+06 2024-01-19 14:52:00+00:00      NaN        NaN
10      carts-db         mem     25  7.884390e+07  8.425316e+07  8.396829e+07 2024-01-19 14:52:00+00:00      NaN        NaN
11      carts-db      socket     25  6.000000e+00  7.000000e+00  7.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN
12     catalogue         cpu     25  1.336217e-01  1.858281e-01  1.833097e-01 2024-01-19 14:52:00+00:00      NaN        NaN
13     catalogue  latency-50     25  3.002940e-03  3.110457e-03  3.095734e-03 2024-01-19 14:52:00+00:00      NaN        NaN
14     catalogue  latency-90     25  4.605291e-03  4.798823e-03  4.772321e-03 2024-01-19 14:52:00+00:00      NaN        NaN
15     catalogue         mem     25  6.176768e+06  6.272137e+06  6.267372e+06 2024-01-19 14:52:00+00:00      NaN        NaN
16     catalogue      socket     25  6.000000e+00  6.000000e+00  6.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN
17     catalogue    workload     25  3.925483e+00  4.323467e+00  4.260883e+00 2024-01-19 14:52:00+00:00      NaN        NaN
18  catalogue-db         cpu     25  1.749846e-01  1.977208e-01  1.954904e-01 2024-01-19 14:52:00+00:00      NaN        NaN
19  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 14:52:00+00:00      NaN        NaN```
```

In [ ]:
# In[3]:

# Identify anomalies by comparing filtered data against global P95 thresholds
anomalies = []

for index, row in filtered_agg.iterrows():
    # Get the global P95 threshold for the current component-KPI
    p95_threshold = row['p95_value']
    
    # Filter the original metric data for the current component-KPI
    current_kpi_data = metric_df[(metric_df['cmdb_id'] == row['cmdb_id']) & 
                                  (metric_df['kpi_name'] == row['kpi_name'])]
    
    # Identify anomalies that exceed the P95 threshold
    current_anomalies = current_kpi_data[current_kpi_data['value'] > p95_threshold]
    
    # Collect anomaly details
    for _, anomaly in current_anomalies.iterrows():
        anomalies.append({
            'cmdb_id': row['cmdb_id'],
            'kpi_name': row['kpi_name'],
            'anomaly_value': anomaly['value'],
            'timestamp': anomaly['timestamp'],
            'severity': anomaly['value'] - p95_threshold  # Severity as the difference from P95
        })

# Create a DataFrame for anomalies
anomalies_df = pd.DataFrame(anomalies)

# Summarize the anomalies
anomaly_summary = anomalies_df.groupby(['cmdb_id', 'kpi_name']).agg(
    anomaly_count=('anomaly_value', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    max_severity=('severity', 'max')
).reset_index()

# Display the anomaly summary
anomaly_summary_result = anomaly_summary.head(20)
anomaly_summary_result

```
Out[3]:
```
The anomaly detection results indicate that several KPIs across different components experienced anomalies during the incident window. Each identified anomaly exceeded the global P95 thresholds. The summary includes:

- For the "carts" component, anomalies were detected in CPU, disk I/O, memory, and latency metrics, with the maximum severity reaching significant values, particularly in disk I/O.
- The "carts-db" component also showed anomalies in CPU, disk I/O, and memory.
- The "catalogue" component had similar findings, with anomalies in CPU, memory, and latency metrics.
- Additionally, the "front-end" component exhibited anomalies in CPU, error counts, and latency.

Each KPI had two anomalies, with the earliest timestamps and maximum severity values recorded. This information is crucial for diagnosing potential issues within the system during the specified time frame.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  anomaly_count        earliest_timestamp   max_severity
0          carts         cpu              2 2024-01-19 15:08:00+00:00       0.245357
1          carts      diskio              2 2024-01-19 14:52:00+00:00   57147.612098
2          carts  latency-50              2 2024-01-19 15:13:00+00:00       0.000947
3          carts  latency-90              2 2024-01-19 15:13:00+00:00       0.018159
4          carts         mem              2 2024-01-19 15:13:00+00:00  504324.096000
5          carts      socket              2 2024-01-19 15:05:00+00:00       0.380000
6          carts    workload              2 2024-01-19 15:01:00+00:00       0.169120
7       carts-db         cpu              2 2024-01-19 15:15:00+00:00       0.113662
8       carts-db      diskio              2 2024-01-19 15:02:00+00:00   25803.427312
9       carts-db         mem              2 2024-01-19 15:15:00+00:00  284876.800000
10     catalogue         cpu              2 2024-01-19 14:54:00+00:00       0.002518
11     catalogue  latency-50              2 2024-01-19 15:11:00+00:00       0.000015
12     catalogue  latency-90              2 2024-01-19 15:11:00+00:00       0.000027
13     catalogue         mem              2 2024-01-19 15:06:00+00:00    4765.013333
14     catalogue    workload              2 2024-01-19 15:01:00+00:00       0.062583
15  catalogue-db         cpu              2 2024-01-19 15:08:00+00:00       0.002230
16  catalogue-db         mem              2 2024-01-19 14:59:00+00:00     709.973333
17     front-end         cpu              2 2024-01-19 14:54:00+00:00       0.031389
18     front-end       error              2 2024-01-19 15:12:00+00:00       0.031273
19     front-end  latency-50              2 2024-01-19 15:13:00+00:00       0.000256```
```

In [ ]:
# In[4]:

# Filter the trace data for the 'carts' component and the disk I/O anomalies
diskio_anomalies = anomaly_summary[(anomaly_summary['cmdb_id'] == 'carts') & 
                                    (anomaly_summary['kpi_name'] == 'diskio')]

# Extract the relevant timestamps for the anomalies
anomaly_timestamps = diskio_anomalies['earliest_timestamp'].values

# Filter the trace data for the relevant timestamps
trace_anomalies = trace_df[trace_df['cmdb_id'] == 'carts']
trace_anomalies = trace_anomalies[trace_anomalies['timestamp'].isin(anomaly_timestamps)]

# Summarize the faults associated with the trace data
fault_summary = trace_anomalies.groupby('trace_name').agg(
    fault_count=('value', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    max_value=('value', 'max')
).reset_index()

# Display the fault summary for the 'carts' component related to disk I/O anomalies
fault_summary_result = fault_summary.head(20)
fault_summary_result

```
Out[4]:
```
The analysis of the trace data for the 'carts' component, specifically focusing on the disk I/O anomalies, did not yield any associated faults. This indicates that there were no relevant trace records linked to the detected disk I/O anomalies during the specified time frame. As a result, no counts of faults, earliest timestamps, or severity values could be identified from the trace data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [trace_name, fault_count, earliest_timestamp, max_value]
Index: []```
```

In [ ]:
# In[5]:

# Define the incident window for log analysis
log_incident_start = pd.to_datetime('2024-01-19 14:49:49', utc=True)
log_incident_end = pd.to_datetime('2024-01-19 15:19:49', utc=True)

# Filter the log data for the 'carts' component within the incident window
log_filtered = log_df[(log_df['cmdb_id'] == 'carts') & 
                       (log_df['timestamp'] >= log_incident_start) & 
                       (log_df['timestamp'] <= log_incident_end)]

# Summarize the relevant log entries
log_summary = log_filtered.groupby('log_name').agg(
    log_count=('value', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    notable_messages=('value', 'first')  # Assuming the first value is notable
).reset_index()

# Display the log summary for the 'carts' component
log_summary_result = log_summary.head(20)
log_summary_result

```
Out[5]:
```
The analysis of the log data for the 'carts' component during the incident window revealed the following:

- There were 23 entries recorded for both error counts and row counts.
- The earliest log entry for both types occurred at 2024-01-19 14:53:00 UTC.
- The notable messages included an error count of 0 and a row count of 50.

These log entries provide insight into the system's behavior during the incident, indicating that while there were no errors reported, there were still significant events logged.

The original code execution output of IPython Kernel is also provided below for reference:

log_name  log_count        earliest_timestamp  notable_messages
0  log.error_count         23 2024-01-19 14:53:00+00:00                 0
1    log.row_count         23 2024-01-19 14:53:00+00:00                50```
```